**Problem 7**

School administrators study the attendance behavior of high-school juniors at two schools. Predictors of the number of days of absence include the type of program in which the student is enrolled and a standardized math test score. The dataset is taken from:

UCLA Negative Binomial Dataset

Fit the negative binomial generalized linear model (GLM) to identify the factors associated with the number of days absent by students. Interpret the results.

In [1]:
import pandas as pd
import numpy as np

import statsmodels.api as sm
import statsmodels.formula.api as smf

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error

In [2]:
url = "https://stats.idre.ucla.edu/stat/stata/dae/nb_data.dta"

df = pd.read_stata(url)

In [3]:
df.head()

,id,gender,math,daysabs,prog
0,1001.0,male,63.0,4.0,2.0
1,1002.0,male,27.0,4.0,2.0
2,1003.0,female,20.0,2.0,2.0
3,1004.0,female,16.0,3.0,2.0
4,1005.0,female,2.0,3.0,2.0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 314 entries, 0 to 313
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype   
---  ------   --------------  -----   
 0   id       314 non-null    float32 
 1   gender   314 non-null    category
 2   math     314 non-null    float32 
 3   daysabs  314 non-null    float32 
 4   prog     314 non-null    float32 
dtypes: category(1), float32(4)
memory usage: 5.5 KB


In [5]:
#Convert Program to Category
df['prog'] = df['prog'].astype('category')

In [6]:
#Exploratory Data Analysis
#Summary Statistics
df.describe()

,id,math,daysabs
count,314.000000,314.000000,314.000000
mean,1575.910889,48.267517,5.955414
std,502.314789,25.362385,7.036952
min,1001.000000,1.000000,0.000000
25%,1079.250000,28.000000,1.000000
50%,1158.500000,48.000000,4.000000
75%,2077.750000,70.000000,8.000000
max,2157.000000,99.000000,35.000000


In [7]:
#Check Overdispersion
print(
    "Mean:",
    df['daysabs'].mean()
)

print(
    "Variance:",
    df['daysabs'].var()
)

Mean: 5.955414
Variance: 49.518699645996094


since Variance > Mean
so overdispersion exists.

This justifies using Negative Binomial regression instead of Poisson regression.

In [8]:
#Fit Negative Binomial Model

#Model:daysabs∼prog+math
model = smf.glm(
    formula='daysabs ~ prog + math',
    data=df,
    family=sm.families.NegativeBinomial()
).fit()

/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


In [9]:
print(model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                daysabs   No. Observations:                  314
Model:                            GLM   Df Residuals:                      310
Model Family:        NegativeBinomial   Df Model:                            3
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -865.68
Date:                Sun, 10 May 2026   Deviance:                       350.98
Time:                        12:03:50   Pearson chi2:                     331.
No. Iterations:                     6   Pseudo R-squ. (CS):             0.1926
Covariance Type:            nonrobust                                         
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept       2.6150      0.200     13.054      

**Interpretation of Negative Binomial Regression Results**

The negative binomial regression model examines how program type and math score affect the number of days absent by students.

The overall model appears meaningful, with a pseudo R² of 0.193, indicating that the predictors explain part of the variation in absenteeism.

**Interpretation of Predictors**

**Math Score**

Coefficient = -0.0060
p-value = 0.018

Math score is statistically significant.

The negative coefficient indicates that students with higher math scores tend to have fewer days absent.

**Program Type**

Reference category = Program 1

**Program 2**

Coefficient = -0.4408
p-value = 0.017

Program 2 is statistically significant.

Students in Program 2 are expected to have fewer absences compared to students in Program 1.

**Program 3**

Coefficient = -1.2786
p-value < 0.001

Program 3 is highly significant.

Students in Program 3 are expected to have substantially fewer absences compared to students in Program 1.

**Interpretation of Coefficients**

Negative Binomial regression uses a log link:

log(μ)=β0+ β1X1 +⋯


Negative coefficients reduce the expected count of absences.

**Overall Conclusion**

The analysis suggests that:

higher math scores are associated with lower absenteeism,
and students enrolled in Programs 2 and 3 tend to have significantly fewer absences than students in the reference program.